# 12 · Test final (2024-25): qué pasó con cada modelo (Sebastián, B)

Este notebook **solo lee** los CSV que dejó la corrida única del test (`docs/resultados_test.csv`,
`docs/bootstrap_test.csv`, `docs/confusion_test.csv`, `docs/resultados_test_meta.json`) y los de
validación. No reentrena ni evalúa nada: los resultados de test se reportan tal cual.

In [ ]:
import sys; sys.path.append("..")
import json
import pandas as pd

from src.evaluation.figuras import fig_comparacion_modelos

FINAL = "logistica_sin_redundancia"
meta = json.load(open("../docs/resultados_test_meta.json", encoding="utf-8"))
print(f"Corrida de test: {meta['fecha']} · commit {meta['commit']} · modo {meta['modo']} · "
      f"n = {meta['n_evaluacion']} partidos de {meta['temporadas_evaluacion']}")
print("Modelo final:", meta["modelo_final"])
assert meta["modo"] == "test" and meta["modelo_final"] == FINAL

val = pd.read_csv("../docs/resultados_validacion.csv").set_index("modelo")
test = pd.read_csv("../docs/resultados_test.csv").set_index("modelo")
boot = pd.read_csv("../docs/bootstrap_test.csv").set_index("modelo")
conf = pd.read_csv("../docs/confusion_test.csv", index_col=0)
assert set(val.index) == set(test.index)

## a. Validación (2023-24) vs. test (2024-25)

Log loss (menor es mejor); `diferencia` = test − validación.

In [ ]:
tabla = pd.DataFrame({"ll_validacion": val["log_loss"], "ll_test": test["log_loss"]})
tabla["diferencia"] = tabla["ll_test"] - tabla["ll_validacion"]
tabla["acc_validacion"] = val["accuracy"]
tabla["acc_test"] = test["accuracy"]
tabla = tabla.sort_values("ll_test")
print("¿Empeoran todos (incluidas las cuotas y el baseline)?", bool((tabla["diferencia"] > 0).all()))
tabla.round(4)

## b. «% del camino recorrido» entre el baseline y las cuotas

`(LL_baseline − LL_modelo) / (LL_baseline − LL_cuotas)`: 0 % = tan malo como adivinar con las
frecuencias, 100 % = tan bueno como las cuotas. Normaliza por la dificultad de cada temporada.

In [ ]:
def pct_camino(res):
    base, cuotas = res.loc["baseline_frecuencias", "log_loss"], res.loc["cuotas_mercado", "log_loss"]
    return (base - res["log_loss"]) / (base - cuotas)

camino = pd.DataFrame({"validacion": pct_camino(val), "test": pct_camino(test)})
camino["diferencia"] = camino["test"] - camino["validacion"]
camino = camino.drop(index=["baseline_frecuencias", "cuotas_mercado"]).sort_values("test", ascending=False)
(camino * 100).round(1).rename(columns=lambda c: c + " (%)")

In [ ]:
# Distancia baseline → cuotas en cada conjunto: la «regla» con la que se mide el camino
for nombre, res in [("validación", val), ("test", test)]:
    print(f"{nombre:10s} baseline {res.loc['baseline_frecuencias','log_loss']:.4f} → cuotas "
          f"{res.loc['cuotas_mercado','log_loss']:.4f}  (distancia {res.loc['baseline_frecuencias','log_loss'] - res.loc['cuotas_mercado','log_loss']:.4f})")

## c. Bootstrap pareado del test contra el modelo final

`diferencia` = LL del modelo − LL del modelo final (positivo = peor que el final). Es significativa si el IC 95 % no contiene el 0.

In [ ]:
b = boot.sort_values("diferencia")
b["lectura"] = [("peor, significativo" if r.significativa and r.diferencia > 0 else
                 "mejor, significativo" if r.significativa else "no distinguible")
                for r in b.itertuples()]
b.round(4)

## d. Matriz de confusión del test, modelo final

Filas = resultado real, columnas = predicción (la clase de mayor probabilidad).

In [ ]:
print("Accuracy:", round(sum(conf.values[i, i] for i in range(3)) / conf.values.sum(), 4),
      "(coincide con resultados_test.csv:", round(test.loc[FINAL, "accuracy"], 4), ")")
print("Veces que predijo empate:", int(conf["pred_D"].sum()), "de", int(conf.values.sum()))
conf

## Figura para la feria

In [ ]:
peores = set(boot.index[boot["significativa"] & (boot["diferencia"] > 0)]) - {"baseline_frecuencias"}
fig = fig_comparacion_modelos(
    test.reset_index(), "../docs/fig_comparacion_test.png", final=FINAL, peores_signif=peores,
    subtitulo="Test 2024-25 · 380 partidos de la Premier · el eje empieza en {x0}",
    etiqueta_x="Log loss en test 2024-25 (menor es mejor)")
print("Con asterisco:", sorted(peores))

## e. Conclusión

Los números salen de las tablas de arriba (nada se reentrenó).

* **Todos empeoran, hasta las cuotas.** Entre validación y test sube el log loss de los 8 modelos:
  las cuotas pasan de 0.909 a 0.971 y el baseline de 1.054 a 1.081. Eso apunta a que 2024-25 fue una
  temporada menos predecible (más sorpresas), no a que el modelo final esté sobreajustado.
* **El modelo final mantuvo su «% del camino».** Recorrió 88.6 % de la distancia entre el baseline y
  las cuotas en validación y 89.3 % en test: no perdió terreno frente a las cuotas. Los árboles sí
  lo perdieron (`xgboost` 82.9 % → 76.5 %, `random_forest` 78.0 % → 75.7 %).
* **El ranking de familias se mantiene:** arriba los modelos lineales y Poisson
  (`poisson_elo`, `logistica_elo`, `logistica_sin_redundancia`, `logistica_todas`) y abajo los
  árboles (`xgboost`, `random_forest`), que son los únicos con diferencia significativa frente al
  modelo final. Las cuotas siguen primeras, pero su ventaja sobre el modelo final tampoco es
  significativa (IC incluye 0).
* **El modelo final quedó prácticamente empatado con `poisson_elo` y `logistica_elo`**
  (diferencias de 0.0016 y 0.0006 de log loss, con IC que incluyen el 0). En test no fue el de menor
  log loss, pero tampoco se puede afirmar que otro sea mejor. **No se cambia**: se eligió el
  2026-10-18, antes de ver el test, y elegir ahora el que salió mejor en test sería usar el test para
  seleccionar y dejaría de ser una evaluación honesta.
* **Limitación:** el modelo final nunca predice empate (0 de 380); las probabilidades de empate
  existen pero nunca son la más alta. Es normal en este problema, pero conviene decirlo.